In [1]:
import pandas as pd
import re

# 입력 (submission template)
in_path = r"C:\Users\Kunny\Research\Project\BiConVarNet\ATP7B\atp7bsubmissiontemplate.csv"
out_path = r"C:\Users\Kunny\Research\Project\BiConVarNet\ATP7B\ATP7B_variants_formatted.tsv"

# UniProt ID / 구조 파일
UNIPROT_ID = "P35670"  # TSC2
STRUCT_FILE = "AF-P35670-F1-model_v4.pdb"

# 아미노산 약어 (3→1)
AA3_TO_AA1 = {
    'Ala':'A','Cys':'C','Asp':'D','Glu':'E','Phe':'F',
    'Gly':'G','His':'H','Ile':'I','Lys':'K','Leu':'L',
    'Met':'M','Asn':'N','Pro':'P','Gln':'Q','Arg':'R',
    'Ser':'S','Thr':'T','Val':'V','Trp':'W','Tyr':'Y',
    'Ter':'*'  # nonsense
}

def parse_variant(var_str):
    # e.g. "p.Ala1617Cys"
    if not var_str.startswith("p."):
        return None, None, None
    var_str = var_str[2:]
    match = re.match(r"([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2}|\*)", var_str)
    if not match:
        return None, None, None
    wt3, pos, mut3 = match.groups()
    wt = AA3_TO_AA1.get(wt3, None)
    mut = AA3_TO_AA1.get(mut3, None)
    return wt, int(pos), mut

# 데이터 불러오기
df = pd.read_csv(in_path)

records = []
for v in df["Variant"]:
    wt, pos, mut = parse_variant(v)
    if wt is None or mut is None:
        continue
    records.append({
        "UniProtID": UNIPROT_ID,
        "MutPos": pos,
        "WT": wt,
        "Mut": mut,
        "Label": "NA",  # 라벨 없음
        "StructureFile": STRUCT_FILE,
        "MutPos(pdb)": pos
    })

out_df = pd.DataFrame(records)

# 위치 기준 정렬
out_df = out_df.sort_values(by=["MutPos", "WT", "Mut"]).reset_index(drop=True)

# 저장
out_df.to_csv(out_path, sep="\t", index=False)
print(f"✅ Saved formatted file: {out_path}, shape={out_df.shape}")


✅ Saved formatted file: C:\Users\Kunny\Research\Project\BiConVarNet\ATP7B\ATP7B_variants_formatted.tsv, shape=(15072, 7)
